In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import tqdm
from thefuzz import fuzz, process
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [2]:
import sys
print(sys.path[0])


c:\Users\liber\OneDrive\Desktop\Università PT2\ML\Project


In [3]:
url = "https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/main/train.csv"

df = pd.read_csv("train.csv")
df.head()


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


# Data Exploration 

In [4]:
print(df.shape)

(75973, 14)


In [5]:
#df.carID.count() # No duplicates for CarID

In [6]:
df.describe()

,carID,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,75973.000000,74482.000000,75973.000000,74510.000000,68069.000000,68047.000000,74457.000000,74449.000000,74423.000000,74425.0
mean,37986.000000,2017.096611,16881.889553,23004.184088,120.329078,55.152666,1.660136,64.590667,1.994580,0.0
std,21931.660338,2.208704,9736.926322,22129.788366,65.521176,16.497837,0.573462,21.021065,1.472981,0.0
min,0.000000,1970.000000,450.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,18993.000000,2016.000000,10200.000000,7423.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,37986.000000,2017.000000,14699.000000,17300.000000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,56979.000000,2019.000000,20950.000000,32427.500000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,75972.000000,2024.121759,159999.000000,323000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


In [7]:
df.dtypes

carID               int64
Brand              object
model              object
year              float64
price               int64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners    float64
hasDamage         float64
dtype: object

In [8]:
df.isna().sum()

carID                0
Brand             1521
model             1517
year              1491
price                0
transmission      1522
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64

# Data Cleaning

## Cleaning text columns
### Resolving Spelling Issues in the Text columns
When exploring the data we see that there a multiple errors with the spelling of the Brand, model, transmission, fuelType columns. In the next section we will try to resolve that and create a coherent naming.

In [9]:
df["Brand"] = df["Brand"].str.lower().str.strip() # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column

df["model"] = df["model"].str.lower().str.strip()
df["transmission"] = df["transmission"].str.lower().str.strip()
df["fuelType"] = df["fuelType"].str.lower().str.strip()

df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN") # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)

In [10]:
# Optional display block, commented for compactnes
# Will show all the unqiue values for the text columns
"""print(df["Brand"].unique())
print("\n------------------------------------ \n")
print(df["model"].unique())
print("\n------------------------------------ \n")
print(df["transmission"].unique())
print("\n------------------------------------ \n")
print(df["fuelType"].unique())"""

'print(df["Brand"].unique())\nprint("\n------------------------------------ \n")\nprint(df["model"].unique())\nprint("\n------------------------------------ \n")\nprint(df["transmission"].unique())\nprint("\n------------------------------------ \n")\nprint(df["fuelType"].unique())'

#### Brands

We decided to do "manual" brand mapping because it gives the biggest controll factor while the number of values is managable. 
Also with some of the brand names beeing very short (e.g. vw) fuzzy algorithms would perform with reduced accuracy

In [11]:
brand_mapping = {
    "vw": "vw",
    "v": "vw",
    "w": "vw",
    
    "toyota": "toyota",
    "toyot": "toyota",
    "oyota": "toyota",
    
    "audi": "audi",
    "aud": "audi",
    "udi": "audi",
    "ud": "audi",
    
    "ford": "ford",
    "for": "ford",
    "ord": "ford",
    "or": "ford",
    
    "bmw": "bmw",
    "bm": "bmw",
    "mw": "bmw",
    
    "skoda": "skoda",
    "skod": "skoda",
    "koda": "skoda",
    "kod": "skoda",
    
    "opel": "opel",
    "ope": "opel",
    "pel": "opel",
    "pe": "opel",
    
    "mercedes": "mercedes",
    "mercede": "mercedes",
    "ercedes": "mercedes",
    "ercede": "mercedes",
    
    "hyundai": "hyundai",
    "hyunda": "hyundai",
    "yundai": "hyundai",
    "yunda": "hyundai"
}

df["Brand"] = df["Brand"].map(brand_mapping)

#### Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)

In [12]:
models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]

short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
short_models = list(set(short_models)) # get unique short model names as a list

transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]

In [13]:
# Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz

for i in range(len(df)): 
    if len(df.model[i]) > 2: # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
        df.loc[i, "model"] = process.extractOne(df.model[i], models)[0] # [0] because we get the name and score as a return -> score used for debugging
    elif len(df.model[i]) == 2: # Use the short names list for comparisons if the model names are 2 letters
        df.loc[i, "model"] = process.extractOne(df.model[i], short_models)[0]
    else: # We can define models with only one letter
        df.loc[i, "model"] = "NaN"

    df.loc[i, "transmission"] = process.extractOne(df.transmission[i], transmission_types)[0]
    df.loc[i, "fuelType"] = process.extractOne(df.fuelType[i], fuel_types)[0]

In [14]:
# Convert the str NaN values back to pd.NA for easier further processing and readability

df["model"] = df["model"].replace("NaN", pd.NA)
df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)

#### Interpolate missing Brand names

In [15]:
brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()) # Get the most frequent brand for each model -> returns df with model and brand
df = pd.merge(df, brand_models, on="model") # add the model and brand df to our main df (onyl add the brand columns, join on model)

df.drop('Brand_x', axis=1, inplace=True) # remove the old brand column
df = df.rename(columns={"Brand_y": "Brand"}) # rename new column
cols = ['carID', 'Brand', 'model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage'] # rearange column order
df = df[cols]

#### Interpolate missing transimission names
Five of the car models in our dataset where only produced with one transmission type but contained missing values in our dataset. For these models we can fill in the missing values as we know which transmission type it should be. This fixes the NA for 140 values.

In [16]:
#transmission_models = df.groupby("model")["transmission"].unique() #.agg(lambda x: x.mode())

"""adam: manual
camry: automatic 
ka: manual
m6: semi-auto
puma: manual"""

df.loc[df["model"]== "adam", "transmission"] = "manual"
df.loc[df["model"]== "camry", "transmission"] = "automatic"
df.loc[df["model"]== "ka", "transmission"] = "manual"
df.loc[df["model"]== "m6", "transmission"] = "semi-auto"
df.loc[df["model"]== "puma", "transmission"] = "manual"

## Cleaning numeric columns

In [17]:
df.year = df.year.round(0)
df.year =  pd.to_datetime(df["year"])
df.price = df.price.round(0) # Only integer prices, shouldn't change much
df.mileage = abs(df.mileage.round(0))
df.tax = abs(df.tax).round(0) # Turn negative values into positve
df.mpg = abs(df.mpg.round(1)) # Leave one after comma digit
df.previousOwners = abs(df.previousOwners.round())

### Removing unecessary comma digits
For most of the numerical columns we have entries with unnecessary after comma numbers. For example 2011.2108... we remove those after additional digits as we assume they are caused by errors and not important information. 
We do the same for negative values, if present. As we believe those are created by typos or system failures (e.g. taxes entered as negative number could mean the employee thought a negative number is required because taxes are deducted)

In [18]:
#df["paintQuality%"].sort_values().unique()

In [19]:
# Check number of of values impacted
# values < 4: 311 
# values > 100: 325
# Check if they are typos, based on mean values with and without wrong values -> no clear difference 
# Removing the values because of the low number of values impacted

In [20]:
#df2 = df.query("`paintQuality%` < 4 or `paintQuality%` > 100")
#df2.groupby("paintQuality%")[["price", "mileage"]].agg(["mean", "median", "count"]).round(2)

In [21]:
#df_clean = df.query("`paintQuality%` > 4 and `paintQuality%` < 100")
#df_clean.groupby("paintQuality%")[["price", "mileage"]].agg(["mean", "median", "count"]).round(2)

In [22]:
#df.shape[0]  -df2.shape[0]

In [23]:
# Filter the wrong values
df = df.query("`paintQuality%` > 4 and `paintQuality%` < 100").copy()

### Further numerical cleaning plan


### hasDamage column
This is a column that is filled by the customer prior to inspection. If the car has no damage the customer fills it in, the other values are left empty. Since it is impossible to determine the level of damage from all the other datapoints we decided to make changes to the column hasDamage. Mainly we change the purpose of the column to assesing if the customer stated that his car has no damage, in this case true (previous 0), else fales (previous missing).

In [24]:
df.hasDamage = df["hasDamage"].fillna(1) # We replace the missing values in hasDamage with 1 as missing values means the customer left the field empty
# Mean and Median very similar for both Damage values 
df.groupby("hasDamage")[["price", "year", "mileage", "paintQuality%"]].agg(["mean", "median"]).round(2).head(2)

price                                   year  \
               mean   median                          mean   
hasDamage                                                    
0.0        16872.13  14698.0 1970-01-01 00:00:00.000002017   
1.0        16900.02  14809.0 1970-01-01 00:00:00.000002017   

                                          mileage          paintQuality%  \
                                 median      mean   median          mean   
hasDamage                                                                  
0.0       1970-01-01 00:00:00.000002017  23444.90  17525.0         64.59   
1.0       1970-01-01 00:00:00.000002017  23324.71  16537.5         64.55   

                  
          median  
hasDamage         
0.0         65.0  
1.0         65.0

In [25]:
# Create the new column
df["stated_no_damage"] = ~df["hasDamage"].astype(bool)

In [26]:
df = df.drop(["hasDamage"], axis=1)

### Removing Columns -> move this after to interpolation as we maybe can use some of the values during interpolation

In [27]:
df.shape

(72047, 14)

In [28]:
# Remove values which we cant interpolate reliably
df = df[df.model.notna()]
df = df[df.year.notna()]
df = df[df.mileage.notna()]
df = df[df.previousOwners.notna()]

In [29]:
df.shape

(67853, 14)

In [30]:
# Remove values which could be interpolated with mode, but would increase bias in the data
df = df[df.transmission.notna()] # Missing values 2078
df = df[df.fuelType.notna()] # Missing values 1532
#df = df[df.engineSize.notna()] # Missing values 1411
df = df[df["paintQuality%"].notna()] # Missing values 1403

In [31]:
df.shape
# Share of data removed 14,19% 

(64398, 14)

### Tax Column
After comparing the mean and median tax statistics for the car models by year we decided to interpolate the missing values using the mean tax value grouped by model, year, transmission, fuel. We think this is a good approaximation of the expected tax amount of that specific car as the tax is based on emissions which vary depending on the factors defined in our grouping methode.

In [32]:
df.groupby(["model", "year", "transmission", "fuelType"])["tax"].agg(["mean", "median", "count"]).round(2).head(2)

mean  median  \
model    year                          transmission fuelType                  
1 series 1970-01-01 00:00:00.000002001 manual       petrol    125.0   125.0   
         1970-01-01 00:00:00.000002004 manual       diesel    200.0   200.0   

                                                              count  
model    year                          transmission fuelType         
1 series 1970-01-01 00:00:00.000002001 manual       petrol        1  
         1970-01-01 00:00:00.000002004 manual       diesel        1

In [33]:
df["tax"] = df["tax"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["tax"].transform("mean")).round(2)

### mpg Column

In [34]:
df.groupby(["model", "year", "transmission", "fuelType"])["mpg"].agg(["mean", "median", "count"]).round(2).head(2)

mean  median  \
model    year                          transmission fuelType                 
1 series 1970-01-01 00:00:00.000002001 manual       petrol    53.3    53.3   
         1970-01-01 00:00:00.000002004 manual       diesel    49.6    49.6   

                                                              count  
model    year                          transmission fuelType         
1 series 1970-01-01 00:00:00.000002001 manual       petrol        1  
         1970-01-01 00:00:00.000002004 manual       diesel        1

In [35]:
df["mpg"] = df["mpg"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["mpg"].transform("mean")).round(2)

## Engine Size

In [36]:
df["engineSize"] = df["engineSize"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["engineSize"].transform("mean")).round(2)

In [37]:
df.isna().sum()

carID                0
Brand                0
model                0
year                 0
price                0
transmission         0
mileage              0
fuelType             0
tax                 46
mpg                 45
engineSize          27
paintQuality%        0
previousOwners       0
stated_no_damage     0
dtype: int64

In [38]:
df = df[df.tax.notna()] # Remove values that couldn't be interpolated
df = df[df.mpg.notna()]
df = df[df.engineSize.notna()]

Brand -> Group by the model and Brand <br>
model -> remove (no clear identifcation thorugh: mpg, engineSize, year, transimission possible) <br>
year -> remove (no interpolation possible, car models where build accross multiple years) <br>
price -> no missing values <br>
transmission -> removed <br>
mileage -> remove (assuemd high volatility for years and mileage) <br>
fuelType ->Removed <br>
tax -> grouped by model, year, transmission, fule mean<br>
mpg -> grouped by model, year, transmission, fule mean <br>
engineSize -> grouped by model, year, transmission, fule mean <br>
paintQuality% -> TBD <br>
previousOwners -> Remove? <br>
hasDamage -> replace Missing values with 1<br>

In [39]:
# Todos
# Turn finished preprocessing into a function (seperate processing for training and test data)


In [40]:
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 64303 entries, 0 to 74253
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   carID             64303 non-null  int64         
 1   Brand             64303 non-null  object        
 2   model             64303 non-null  object        
 3   year              64303 non-null  datetime64[ns]
 4   price             64303 non-null  int64         
 5   transmission      64303 non-null  object        
 6   mileage           64303 non-null  float64       
 7   fuelType          64303 non-null  object        
 8   tax               64303 non-null  float64       
 9   mpg               64303 non-null  float64       
 10  engineSize        64303 non-null  float64       
 11  paintQuality%     64303 non-null  float64       
 12  previousOwners    64303 non-null  float64       
 13  stated_no_damage  64303 non-null  bool          
dtypes: bool(1), datetime64[ns](

carID               0
Brand               0
model               0
year                0
price               0
transmission        0
mileage             0
fuelType            0
tax                 0
mpg                 0
engineSize          0
paintQuality%       0
previousOwners      0
stated_no_damage    0
dtype: int64

# Preprocessing, train and selection

Drop 1-hot with the minority class

~~check the prperocessing if it's ok with the data leakage. Sava data from xtrain, apply to xval~~

Try cross val. Try basic radom search and gridsearch

dont remove 1970, use standard/robust scaler instead of minmax

MLP design, basic should have input layer = feature input, 1 hidden layer, single perceptron output layer

Apply other metrics shown in class

Models:

1. Linear Regression -> 4866.478
3. Ridge -> 4865.080
4. Lasso -> 4863.857
2. ElasticNet -> 8280.295
5. Adaline -> ask professor if sgd algos can be used
6. SVM -> 8913.638
7. Random Forest -> 3218.916
8. Single Tree -> 4390.532
9. KNN Regressor -> 4589.610
10. SGD
10. MLP


1-5 Linear

In [41]:
df["year"] = df["year"].astype(str).str[-4:].astype(int)
df["stated_no_damage"] = df["stated_no_damage"].astype(int)
df.set_index("carID", inplace=True)

In [42]:
df_num = df[["year", "price", "mileage", "tax", "mpg", "engineSize","paintQuality%", "previousOwners", "stated_no_damage"]]
df_num

,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage
carID,,,,,,,,,
69512,2016,22290,28421.0,95.95,11.4,2.0,63.0,4.0,1
53000,2019,13790,4589.0,145.00,47.9,1.5,50.0,1.0,1
6366,2019,24990,3624.0,145.00,40.9,1.5,56.0,4.0,1
29021,2018,12500,9102.0,145.00,65.7,1.0,50.0,2.0,1
10062,2019,22995,1000.0,145.00,42.8,1.5,97.0,3.0,1
...,...,...,...,...,...,...,...,...,...
37194,2015,13498,14480.0,125.00,53.3,2.0,78.0,0.0,1
6265,2013,12495,52134.0,200.00,47.9,2.0,38.0,2.0,1
54886,2017,8399,11304.0,145.00,67.0,1.0,57.0,3.0,1


In [43]:
df_cat = df[["Brand", "model", "transmission", "fuelType"]]
df_cat

,Brand,model,transmission,fuelType
carID,,,,
69512,vw,golf,semi-auto,petrol
53000,toyota,yaris,manual,petrol
6366,audi,q2,semi-auto,petrol
29021,ford,fiesta,manual,petrol
10062,bmw,2 series,manual,petrol
...,...,...,...,...
37194,mercedes,c class,manual,petrol
6265,audi,q3,semi-auto,diesel
54886,toyota,aygo,automatic,petrol


In [44]:
freq_brand = df_cat["Brand"].value_counts()
print(freq_brand)

Brand
ford        13966
mercedes    10088
vw           8987
opel         8130
bmw          6375
audi         6231
toyota       3931
skoda        3699
hyundai      2896
Name: count, dtype: int64


In [45]:
model_freq = df_cat["model"].value_counts()
print(model_freq)

model
focus       6040
c class     4582
fiesta      3879
golf        2827
corsa       2008
            ... 
fox            1
200            1
s5             1
escort         1
terracan       1
Name: count, Length: 191, dtype: int64


In [46]:
freq_trans = df_cat["transmission"].value_counts()
print(freq_trans)

transmission
manual       36418
semi-auto    14770
automatic    13115
Name: count, dtype: int64


In [47]:
freq_fuel = df_cat["fuelType"].value_counts()
print(freq_fuel)

fuelType
petrol      35649
diesel      26714
hybrid       1937
electric        3
Name: count, dtype: int64


In [48]:
freq_year = df["year"].value_counts()
print(freq_year)

year
2019    17560
2017    13970
2016     9941
2018     8900
2015     4918
2020     2708
2014     2515
2013     1689
2011      482
2012      404
2010      288
2009      203
2023      186
2008      130
2024      111
2007      105
2005       50
2006       49
2004       27
2003       25
2002       15
2001       12
2000        5
1999        5
1998        2
1996        1
1970        1
1997        1
Name: count, dtype: int64


In [49]:
freq_damage = df["stated_no_damage"].value_counts()
print(freq_damage)

stated_no_damage
1    62979
0     1324
Name: count, dtype: int64


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 64303 entries, 69512 to 15795
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brand             64303 non-null  object 
 1   model             64303 non-null  object 
 2   year              64303 non-null  int64  
 3   price             64303 non-null  int64  
 4   transmission      64303 non-null  object 
 5   mileage           64303 non-null  float64
 6   fuelType          64303 non-null  object 
 7   tax               64303 non-null  float64
 8   mpg               64303 non-null  float64
 9   engineSize        64303 non-null  float64
 10  paintQuality%     64303 non-null  float64
 11  previousOwners    64303 non-null  float64
 12  stated_no_damage  64303 non-null  int64  
dtypes: float64(6), int64(3), object(4)
memory usage: 6.9+ MB


In [51]:
df


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage
carID,,,,,,,,,,,,,
69512,vw,golf,2016,22290,semi-auto,28421.0,petrol,95.95,11.4,2.0,63.0,4.0,1
53000,toyota,yaris,2019,13790,manual,4589.0,petrol,145.00,47.9,1.5,50.0,1.0,1
6366,audi,q2,2019,24990,semi-auto,3624.0,petrol,145.00,40.9,1.5,56.0,4.0,1
29021,ford,fiesta,2018,12500,manual,9102.0,petrol,145.00,65.7,1.0,50.0,2.0,1
10062,bmw,2 series,2019,22995,manual,1000.0,petrol,145.00,42.8,1.5,97.0,3.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,mercedes,c class,2015,13498,manual,14480.0,petrol,125.00,53.3,2.0,78.0,0.0,1
6265,audi,q3,2013,12495,semi-auto,52134.0,diesel,200.00,47.9,2.0,38.0,2.0,1
54886,toyota,aygo,2017,8399,automatic,11304.0,petrol,145.00,67.0,1.0,57.0,3.0,1


In [52]:
df = df.drop(["model"], axis=1)             #for the general model we don't care for the models, it's too many zero columns


In [53]:
def Preprocessing(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    #TODOs Moritz
    
    num_cols = ["year", "mileage", "tax", "mpg", "engineSize",
                "paintQuality%", "previousOwners", "stated_no_damage"]
    cat_cols = ["Brand", "transmission", "fuelType"]
    
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

    encoder = OneHotEncoder(sparse_output=False, dtype=int, handle_unknown='ignore')
    encoded_train = encoder.fit_transform(X_train[cat_cols])
    encoded_test = encoder.transform(X_test[cat_cols])
    
    encoded_train_df = pd.DataFrame(
        encoded_train, 
        columns=encoder.get_feature_names_out(cat_cols), 
        index=X_train.index)
    
    encoded_test_df = pd.DataFrame(
        encoded_test, 
        columns=encoder.get_feature_names_out(cat_cols), 
        index=X_test.index)
    

    X_train_encoded = pd.concat([X_train.drop(columns=cat_cols), encoded_train_df], axis=1)
    X_test_encoded = pd.concat([X_test.drop(columns=cat_cols), encoded_test_df], axis=1)

    X_test_encoded = X_test_encoded[X_train_encoded.columns]
    
    return X_train_encoded, X_test_encoded, scaler, encoder


In [54]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

def cross_validation(X, y, model, n_folds=5, n_bins=10, metrics=None, random_state=42):
    """
    Cross-validation with stratification and preprocessing
    
    Parameters:
    -----------
    X : pd.DataFrame
        Features
    y : pd.Series or np.array
        Target variable
    model : object
        Model 
    n_folds : int
        Folds chosen
    n_bins : int
        bins for our stratification
    metrics : list of tuples
        metrics
        Es: [('MSE', mean_squared_error), ('MAE', mean_absolute_error)]
    random_state : int
        Seed 
        
    Returns:
    --------
    results : dict
        A dictionary:
        - 'fold_results': metrics for every fold
        - 'mean_metrics': mean of the metrics
        - 'std_metrics': standard deviation for the metrics
        - 'trained_models': models
    """
    if metrics is None:
        metrics = [
            ('MSE', mean_squared_error),
            ('RMSE', lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
            ('MAE', mean_absolute_error),
            ('R2', r2_score)
        ]
    

    y_bins = pd.cut(y, bins=n_bins, labels=False)
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    fold_results = {metric_name: [] for metric_name, _ in metrics}
    fold_results_train = {metric_name: [] for metric_name, _ in metrics}  # Per train metrics
    trained_models = []

    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):
        print(f"\n{'='*80}")
        print(f"FOLD {fold_idx}/{n_folds}")
        print(f"{'='*80}")
        
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y.iloc[train_idx] if isinstance(y, pd.Series) else y[train_idx]
        y_val_fold = y.iloc[val_idx] if isinstance(y, pd.Series) else y[val_idx]
        
        print(f"Train size: {len(X_train_fold)}, Validation size: {len(X_val_fold)}")
        
        X_train_processed, X_val_processed, scaler, encoder = Preprocessing(
            X_train_fold, X_val_fold
        )
        

        model_fold = model.__class__(**model.get_params())
        model_fold.fit(X_train_processed, y_train_fold)
        

        y_pred_train = model_fold.predict(X_train_processed)
        y_pred_val = model_fold.predict(X_val_processed)
        
    
        print(f"\n📊 TRAIN Metrics:")
        print("-" * 40)
        for metric_name, metric_func in metrics:
            score_train = metric_func(y_train_fold, y_pred_train)
            fold_results_train[metric_name].append(score_train)
            print(f"  {metric_name:12s}: {score_train:12.4f}")
 
        print(f" VALIDATION Metrics:")
        print("-" * 40)
        for metric_name, metric_func in metrics:
            score_val = metric_func(y_val_fold, y_pred_val)
            fold_results[metric_name].append(score_val)
            print(f"  {metric_name:12s}: {score_val:12.4f}")
        

        trained_models.append({
            'model': model_fold,
            'scaler': scaler,
            'encoder': encoder
        })
    

    mean_metrics = {metric: np.mean(scores) for metric, scores in fold_results.items()}
    std_metrics = {metric: np.std(scores) for metric, scores in fold_results.items()}
    mean_metrics_train = {metric: np.mean(scores) for metric, scores in fold_results_train.items()}
    std_metrics_train = {metric: np.std(scores) for metric, scores in fold_results_train.items()}
    

    print(f"\n\n{'='*80}")
    print("Metrics for each fold")
    print(f"{'='*80}\n")
    

    print(f"{'Metric':<12s} | ", end="")
    for i in range(1, n_folds + 1):
        print(f"Fold {i:>2d}  ", end=" | ")
    print(f"{'Mean':>10s} | {'Std':>10s}")
    print("-" * 80)
    
    for metric_name in fold_results.keys():
        print(f"{metric_name:<12s} | ", end="")
        for score in fold_results[metric_name]:
            print(f"{score:>8.4f}", end=" | ")
        print(f"{mean_metrics[metric_name]:>10.4f} | {std_metrics[metric_name]:>10.4f}")
    
    print(f"\n{'='*80}")
    print("Results on validation set")
    print(f"{'='*80}")
    for metric_name in fold_results.keys():
        print(f"{metric_name:12s}: {mean_metrics[metric_name]:10.4f} (±{std_metrics[metric_name]:8.4f})")
    
    print(f"\n{'='*80}")
    print("Results on training set")
    print(f"{'='*80}")
    for metric_name in fold_results_train.keys():
        print(f"{metric_name:12s}: {mean_metrics_train[metric_name]:10.4f} (±{std_metrics_train[metric_name]:8.4f})")
    
    return {
        'fold_results_val': fold_results,
        'fold_results_train': fold_results_train,
        'mean_metrics_val': mean_metrics,
        'std_metrics_val': std_metrics,
        'mean_metrics_train': mean_metrics_train,
        'std_metrics_train': std_metrics_train,
        'trained_models': trained_models
    }



In [55]:
X = df.drop("price", axis=1)
y = df["price"]


n_bins = 10
y_bins = pd.cut(y, bins=n_bins, labels=False)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=1,
    stratify=y_bins
)

In [56]:
print(f"X_train shape is {X_train.shape}")
print(f"X_val size is {X_val.shape}")

X_train shape is (51442, 11)
X_val size is (12861, 11)


In [60]:
pipe_linear = LinearRegression()

results = cross_validation(
    X=X_train,           
    y=y_train,
    model=pipe_linear,
    n_folds=10,
    n_bins=10,
    random_state=42
)

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(



FOLD 1/10
Train size: 46297, Validation size: 5145

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25015250.4122
  RMSE        :    5001.5248
  MAE         :    3076.8051
  R2          :       0.7377
 VALIDATION Metrics:
----------------------------------------
  MSE         : 25689684.3205
  RMSE        :    5068.4992
  MAE         :    3085.2523
  R2          :       0.7313

FOLD 2/10
Train size: 46297, Validation size: 5145

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25106217.3659
  RMSE        :    5010.6105
  MAE         :    3078.4775
  R2          :       0.7361
 VALIDATION Metrics:
----------------------------------------
  MSE         : 24871807.9159
  RMSE        :    4987.1643
  MAE         :    3075.3451
  R2          :       0.7456

FOLD 3/10
Train size: 46298, Validation size: 5144

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25167049.0616
  RMSE        :    5016.6771
  MAE         : 

In [61]:
X_train_processed, X_val_processed, scaler, encoder = Preprocessing(X_train, X_val)

pipe_linear_final = LinearRegression()  
pipe_linear_final.fit(X_train_processed, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [62]:
y_pred = pipe_linear_final.predict(X_val_processed)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print(f"\n{'='*50}")
print("FINAL VALIDATION RESULTS")
print(f"{'='*50}")
print(f"Validation MSE:  {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


print(f"\nCross-Validation MSE:  {results['mean_metrics_val']['MSE']:.3f}")
print(f"Cross-Validation RMSE: {results['mean_metrics_val']['RMSE']:.3f}")


FINAL VALIDATION RESULTS
Validation MSE:  23744931.593
Validation RMSE: 4872.877

Cross-Validation MSE:  25110972.472
Cross-Validation RMSE: 5010.210


Linear Regression

In [82]:
pipe_linear_regression = Pipeline([("model", LinearRegression())])

In [83]:
pipe_linear_regression.fit(X_train, y_train)
y_pred = pipe_linear_regression.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4822.507


Ridge

In [68]:
pipe_ridge = Pipeline([("model", Ridge())])

In [69]:
pipe_ridge.fit(X_train, y_train)
y_pred = pipe_ridge.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4822.423


Lasso

In [70]:
pipe_lasso = Pipeline([("model", Lasso())])

In [71]:
pipe_lasso.fit(X_train, y_train)
y_pred = pipe_lasso.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4823.400


ElasticNet

In [72]:
pipe_elastic = Pipeline([("model", ElasticNet())])

In [73]:
pipe_elastic.fit(X_train, y_train)
y_pred = pipe_elastic.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 8259.740


SVMRegressor

In [358]:
pipe_SVM = Pipeline([("model", SVR())])

In [361]:
pipe_SVM.fit(X_train, y_train)
y_pred = pipe_SVM.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print(f"Validation MSE: {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


Validation MSE: 79452948.717
Validation RMSE: 8913.638


Random Forest

In [371]:
pipe_rf = Pipeline([("model", RandomForestRegressor())])


In [373]:
pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

#print(f"Validation MSE: {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


Validation RMSE: 3218.916


Single tree

In [374]:
pipe_tree = Pipeline([("model", DecisionTreeRegressor())])

In [375]:
pipe_tree.fit(X_train, y_train)
y_pred = pipe_tree.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4390.532


KNRegressor

In [376]:
pipe_KNR = pipe_tree = Pipeline([("model", KNeighborsRegressor())])

In [377]:
pipe_KNR.fit(X_train, y_train)
y_pred = pipe_KNR.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4589.610


Neural Network



In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.7, shuffle=True)

In [ ]:
model = nn.Sequential(
    nn.Linear
)

TypeError: torch.nn.modules.linear.Linear is not a Module subclass